In [5]:
# SILVER NOTEBOOK — Full Auto Discovery + Cleansing + SCD1 / SCD2
# FIX: Removed incremental filter from Silver.
#      Silver always reads ALL active records from Bronze and applies SCD logic.
#      Incremental filtering belongs in Bronze only (source → Bronze).
#      Silver's job = transform Bronze data into clean versioned Silver tables.

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType, BooleanType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

spark.conf.set("spark.sql.shuffle.partitions", "2")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# ── ONLY EDIT THESE 4 LINES ───────────────────────────────────────────────────
BRONZE_DB   = "LH_Bronze_layer.dbo"
SILVER_DB   = "LH_Silver_layer.dbo"
AUDIT_DB    = "LH_Gold_layer.audit"
DEFAULT_SCD = "SCD1"          # SCD1 or SCD2
# ─────────────────────────────────────────────────────────────────────────────

run_id        = str(uuid.uuid4())
pipeline_name = "Silver_Pipeline"
master_start  = datetime.now()
HIGH          = F.lit("9999-12-31").cast(TimestampType())

print(f"[SILVER] Started   : {master_start}")
print(f"[SILVER] Run ID    : {run_id}")
print(f"[SILVER] Bronze DB : {BRONZE_DB}")
print(f"[SILVER] Silver DB : {SILVER_DB}")
print(f"[SILVER] SCD Mode  : {DEFAULT_SCD}")

# ── Tables with known composite PKs ──────────────────────────────────────────
# Add more entries here if you have other composite PK tables
COMPOSITE_PK_MAP = {
    "productvendor":                         ["ProductID", "BusinessEntityID"],
    "productcosthistory":                    ["ProductID", "StartDate"],
    "productlistpricehistory":               ["ProductID", "StartDate"],
    "productmodelillustration":              ["ProductModelID", "IllustrationID"],
    "productmodelproductdescriptionculture": ["ProductModelID", "ProductDescriptionID", "CultureID"],
    "productproductphoto":                   ["ProductID", "ProductPhotoID"],
    "productdocument":                       ["ProductID", "DocumentNode"],
    "purchaseorderdetail":                   ["PurchaseOrderID", "PurchaseOrderDetailID"],
    "salesorderdetail":                      ["SalesOrderID", "SalesOrderDetailID"],
    "specialofferproduct":                   ["SpecialOfferID", "ProductID"],
    "countryregioncurrency":                 ["CountryRegionCode", "CurrencyCode"],
    "currencyrate":                          ["CurrencyRateID"],
    "personphone":                           ["BusinessEntityID", "PhoneNumber", "PhoneNumberTypeID"],
    "emailaddress":                          ["BusinessEntityID", "EmailAddressID"],
    "businessentityaddress":                 ["BusinessEntityID", "AddressID", "AddressTypeID"],
    "businessentitycontact":                 ["BusinessEntityID", "PersonID", "ContactTypeID"],
}

# ── Audit Schemas ─────────────────────────────────────────────────────────────
AUDIT_SCHEMA = StructType([
    StructField("ROW_ID",        StringType(),    True),
    StructField("RUN_ID",        StringType(),    True),
    StructField("CREATED_DATE",  TimestampType(), True),
    StructField("PIPELINE_NAME", StringType(),    True),
    StructField("SOURCE_TYPE",   StringType(),    True),
    StructField("SOURCE_TABLE",  StringType(),    True),
    StructField("TARGET_TABLE",  StringType(),    True),
    StructField("START_TIME",    TimestampType(), True),
    StructField("END_TIME",      TimestampType(), True),
    StructField("STATUS",        StringType(),    True),
    StructField("STATUS_DESC",   StringType(),    True),
])

COUNT_SCHEMA = StructType([
    StructField("ROW_ID",                             StringType(),    True),
    StructField("RUN_ID",                             StringType(),    True),
    StructField("SOURCE_TYPE",                        StringType(),    True),
    StructField("SOURCE_TABLE",                       StringType(),    True),
    StructField("TARGET_TABLE",                       StringType(),    True),
    StructField("LAYER",                              StringType(),    True),
    StructField("SOUREC_FILE_COUNT",                  StringType(),    True),
    StructField("STAGING_FILE_COUNT",                 StringType(),    True),
    StructField("SOURCE_TABLE_COUNT",                 LongType(),      True),
    StructField("STAGING_TABLE_COUNT",                LongType(),      True),
    StructField("ERROR_COUNT_SOURCE_TO_STAGING_FILE", LongType(),      True),
    StructField("ERROR_COUNT_SOURCE_TO_STAGING",      LongType(),      True),
    StructField("Current_time",                       TimestampType(), True),
])

# ── Helpers ───────────────────────────────────────────────────────────────────
def tbl_exists(t):
    try:
        spark.sql(f"DESCRIBE TABLE {t}")
        return True
    except:
        return False

def audit(tbl, table_name, src, tgt, t0, s, d=""):
    try:
        row = spark.createDataFrame(
            [(str(uuid.uuid4()), run_id, datetime.now(), pipeline_name,
              src, table_name, tgt, t0, datetime.now(), str(s), str(d)[:500])],
            schema=AUDIT_SCHEMA
        )
        for field in AUDIT_SCHEMA.fields:
            row = row.withColumn(field.name, F.col(field.name).cast(field.dataType))
        row.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(tbl)
    except Exception as ae:
        print(f"  [AUDIT WARNING] {ae}")

def count_audit(table_name, src, tgt, src_cnt, stg_cnt, err=0):
    try:
        row = spark.createDataFrame(
            [(str(uuid.uuid4()), run_id, src, table_name, tgt, "SILVER",
              None, None, int(src_cnt), int(stg_cnt), int(err), int(err), datetime.now())],
            schema=COUNT_SCHEMA
        )
        for field in COUNT_SCHEMA.fields:
            row = row.withColumn(field.name, F.col(field.name).cast(field.dataType))
        row.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(f"{AUDIT_DB}.count_log_table")
    except Exception as ce:
        print(f"  [COUNT WARNING] {ce}")

# ── Detect PK for a table ─────────────────────────────────────────────────────
def detect_pks(table_name, columns):
    table_lower = table_name.lower()
    if table_lower in COMPOSITE_PK_MAP:
        # Use known composite PK — filter to only columns that exist in this table
        known = COMPOSITE_PK_MAP[table_lower]
        valid = [k for k in known if k in columns]
        if valid:
            return valid
    # Auto-detect: first column ending with "id" or "ID"
    pk = next((c for c in columns if c.lower().endswith("id")), columns[0])
    return [pk]

# ── SCD1: Upsert — no history kept ───────────────────────────────────────────
def apply_scd1(df, PKS, TGT):
    df = (df
        .withColumn("_silver_effective_from", F.current_timestamp())
        .withColumn("_silver_is_current",     F.lit(True).cast(BooleanType()))
        .withColumn("_silver_run_id",         F.lit(run_id)))

    if tbl_exists(TGT):
        cond = " AND ".join([f"t.`{k}` = s.`{k}`" for k in PKS])
        upd  = {f"`{c}`": f"s.`{c}`" for c in df.columns if c not in PKS}
        ins  = {f"`{c}`": f"s.`{c}`" for c in df.columns}
        (DeltaTable.forName(spark, TGT).alias("t")
            .merge(df.alias("s"), cond)
            .whenMatchedUpdate(set=upd)
            .whenNotMatchedInsert(values=ins)
            .execute())
        print(f"  SCD1 MERGE complete.")
    else:
        df.write.format("delta").mode("overwrite").saveAsTable(TGT)
        print(f"  SCD1 CREATE complete.")

# ── SCD2: Expire old + insert new — full history kept ────────────────────────
def apply_scd2(df, PKS, TGT):
    non_pk_cols = [c for c in df.columns if c not in PKS]
    hash_expr   = F.md5(F.concat_ws("|", *[F.col(c).cast("string") for c in non_pk_cols]))
    sk_expr     = F.md5(F.concat_ws("|", *[F.col(k).cast("string") for k in PKS],
                                    F.current_timestamp().cast("string")))
    df = (df
        .withColumn("_silver_sk",             sk_expr)
        .withColumn("_silver_effective_from", F.current_timestamp())
        .withColumn("_silver_effective_to",   HIGH)
        .withColumn("_silver_is_current",     F.lit(True).cast(BooleanType()))
        .withColumn("_silver_run_id",         F.lit(run_id))
        .withColumn("_row_hash",              hash_expr))

    if not tbl_exists(TGT):
        df.drop("_row_hash").write.format("delta").mode("overwrite").saveAsTable(TGT)
        print(f"  SCD2 initial load complete.")
        return

    existing = (spark.read.format("delta").table(TGT)
                    .filter(F.col("_silver_is_current") == True)
                    .withColumn("_row_hash", hash_expr)
                    .select(*PKS, "_row_hash"))

    changed  = (df.alias("i")
                    .join(existing.alias("e"), PKS, "inner")
                    .filter(F.col("i._row_hash") != F.col("e._row_hash"))
                    .select("i.*"))
    new_rows  = df.alias("i").join(existing.alias("e"), PKS, "left_anti").select("i.*")
    to_insert = changed.union(new_rows).drop("_row_hash")

    if to_insert.count() == 0:
        print(f"  SCD2: No changes detected.")
        return

    expire_keys = spark.createDataFrame([r.asDict() for r in changed.select(*PKS).collect()])
    exp_cond    = " AND ".join([f"t.`{k}` = e.`{k}`" for k in PKS])
    (DeltaTable.forName(spark, TGT).alias("t")
        .merge(expire_keys.alias("e"), f"({exp_cond}) AND t._silver_is_current = true")
        .whenMatchedUpdate(set={
            "_silver_is_current":   F.lit(False),
            "_silver_effective_to": F.current_timestamp()
        }).execute())

    to_insert.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(TGT)
    print(f"  SCD2: Expired old rows, inserted {to_insert.count()} new records.")

# ── Core: Process One Table ───────────────────────────────────────────────────
def process_table(table_name):
    SRC    = f"{BRONZE_DB}.{table_name}"
    TGT    = f"{SILVER_DB}.{table_name}"
    t0     = datetime.now()
    status = "SUCCESS"
    desc   = ""

    print(f"\n  [{table_name}] Reading from : {SRC}")
    print(f"  [{table_name}] Writing to   : {TGT}")

    # ── Step 1: Read ALL rows from Bronze ─────────────────────────────────────
    # NO incremental filter here — Silver always processes the full Bronze table.
    # SCD1/SCD2 MERGE logic handles what is new vs changed vs unchanged.
    try:
        bronze = spark.read.format("delta").table(SRC)
        src_cnt = bronze.count()
        print(f"  [{table_name}] Source rows   : {src_cnt}")
        print(f"  [{table_name}] Columns       : {bronze.columns}")
    except Exception as e:
        audit(f"{AUDIT_DB}.copy_audit_log",     table_name, SRC, TGT, t0, "FAILED", str(e))
        audit(f"{AUDIT_DB}.notebook_audit_log", table_name, SRC, TGT, t0, "FAILED", str(e))
        raise

    # ── Step 2: Detect PKs ────────────────────────────────────────────────────
    PKS = detect_pks(table_name, bronze.columns)
    print(f"  [{table_name}] PKs           : {PKS}")

    # ── Step 3: Empty check ───────────────────────────────────────────────────
    if src_cnt == 0:
        audit(f"{AUDIT_DB}.copy_audit_log",     table_name, SRC, TGT, t0, "NO_DATA", "Bronze table is empty")
        count_audit(table_name, SRC, TGT, 0, 0)
        audit(f"{AUDIT_DB}.notebook_audit_log", table_name, SRC, TGT, t0, "NO_DATA", "Bronze table is empty")
        print(f"  [{table_name}] NO_DATA — Bronze table is empty")
        return "NO_DATA"

    # ── Step 4: Cleanse ───────────────────────────────────────────────────────
    df = bronze
    # Trim whitespace on all string columns
    for field in df.schema.fields:
        if str(field.dataType) == "StringType()":
            df = df.withColumn(field.name, F.trim(F.col(field.name)))
    # Cast date/time columns to TimestampType
    for col_name in df.columns:
        if "date" in col_name.lower() or "time" in col_name.lower():
            try:
                df = df.withColumn(col_name, F.col(col_name).cast(TimestampType()))
            except:
                pass  # skip non-castable columns silently

    # ── Step 5: Deduplicate on PK (safety — Bronze should not have dups) ──────
    ld_col = next((c for c in df.columns if "modified" in c.lower()), None) or \
             next((c for c in df.columns if "date" in c.lower()), None)

    if ld_col:
        window_spec = Window.partitionBy(*PKS).orderBy(F.col(ld_col).desc())
    else:
        window_spec = Window.partitionBy(*PKS).orderBy(F.lit(1))

    df = (df.withColumn("_rn", F.row_number().over(window_spec))
            .filter(F.col("_rn") == 1)
            .drop("_rn"))

    stg_cnt = df.count()
    print(f"  [{table_name}] Rows to write : {stg_cnt}")

    # ── Step 6: Apply SCD ─────────────────────────────────────────────────────
    try:
        if DEFAULT_SCD.upper() == "SCD2":
            apply_scd2(df, PKS, TGT)
        else:
            apply_scd1(df, PKS, TGT)

        spark.sql(f"OPTIMIZE {TGT} ZORDER BY (`{PKS[0]}`)")
        tgt_cnt = spark.sql(f"SELECT COUNT(1) AS c FROM {TGT}").collect()[0]["c"]
        print(f"  [{table_name}] Target rows   : {tgt_cnt}")

    except Exception as e:
        status = "FAILED"
        desc   = str(e)
        audit(f"{AUDIT_DB}.copy_audit_log",     table_name, SRC, TGT, t0, status, desc)
        count_audit(table_name, SRC, TGT, src_cnt, stg_cnt, err=1)
        audit(f"{AUDIT_DB}.notebook_audit_log", table_name, SRC, TGT, t0, status, desc)
        raise

    # ── Step 7: Audit logs ────────────────────────────────────────────────────
    audit(f"{AUDIT_DB}.copy_audit_log",     table_name, SRC, TGT, t0, status, desc)
    count_audit(table_name, SRC, TGT, src_cnt, stg_cnt)
    audit(f"{AUDIT_DB}.notebook_audit_log", table_name, SRC, TGT, t0, status, desc)

    print(f"  [{table_name}] ✔ SUCCESS | src={src_cnt} stg={stg_cnt} tgt={tgt_cnt} | scd={DEFAULT_SCD}")
    return "SUCCESS"

# ── Auto-Discover ALL Tables from Bronze ─────────────────────────────────────
bronze_tables = [
    row.tableName
    for row in spark.sql(f"SHOW TABLES IN {BRONZE_DB}").collect()
]

print(f"\n[SILVER] Tables discovered in {BRONZE_DB}: {len(bronze_tables)}")
for t in bronze_tables:
    print(f"           → {t}")

# ── Run All Tables ────────────────────────────────────────────────────────────
results = []
for idx, table in enumerate(bronze_tables, 1):
    print(f"\n[SILVER] ══════════════════════════════════════════════")
    print(f"[SILVER] [{idx} / {len(bronze_tables)}]  {table}")
    print(f"[SILVER] ══════════════════════════════════════════════")
    try:
        s = process_table(table)
        results.append((table, s, ""))
    except Exception as e:
        results.append((table, "FAILED", str(e)[:200]))
        print(f"  [{table}] ✘ FAILED → {e}")

# ── Final Summary ─────────────────────────────────────────────────────────────
success = sum(1 for r in results if r[1] == "SUCCESS")
failed  = sum(1 for r in results if r[1] == "FAILED")
no_data = sum(1 for r in results if r[1] == "NO_DATA")

print(f"\n[SILVER] ══════════════════════════════════════════════════")
print(f"[SILVER] PIPELINE COMPLETE : {datetime.now()}")
print(f"[SILVER] Total   = {len(results)}")
print(f"[SILVER] Success = {success}")
print(f"[SILVER] Failed  = {failed}")
print(f"[SILVER] No Data = {no_data}")
print(f"[SILVER] ══════════════════════════════════════════════════")
for table, status, err in results:
    icon = "✔" if status == "SUCCESS" else ("−" if status == "NO_DATA" else "✘")
    msg  = f"→ {err}" if err else ""
    print(f"  {icon}  {table:50s}  {status}  {msg}")

if failed:
    raise Exception(f"[SILVER] {failed} table(s) failed. See details above.")

StatementMeta(, cd4a7589-b3e5-4d2f-a253-c9bffedbdb0f, 7, Finished, Available, Finished, False)

[SILVER] Started   : 2026-04-13 02:27:56.554687
[SILVER] Run ID    : 00025fb3-9efb-40fd-b2e7-d6dc9c638e26
[SILVER] Bronze DB : LH_Bronze_layer.dbo
[SILVER] Silver DB : LH_Silver_layer.dbo
[SILVER] SCD Mode  : SCD1

[SILVER] Tables discovered in LH_Bronze_layer.dbo: 17
           → product
           → productcategory
           → productcosthistory
           → productdescription
           → productdocument
           → productinventory
           → productlistpricehistory
           → productmodel
           → productmodelillustration
           → productmodelproductdescriptionculture
           → productphoto
           → productproductphoto
           → productreview
           → productsubcategory
           → productvendor
           → purchaseorderdetail
           → purchaseorderheader

[SILVER] ══════════════════════════════════════════════
[SILVER] [1 / 17]  product
[SILVER] ══════════════════════════════════════════════

  [product] Reading from : LH_Bronze_layer.dbo.product